In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. LOAD THE DATASETS
# ==========================================

# NOTE: Download the dataset from Kaggle and paste your local file paths inside the quotes below
books_path = "HERE_GOES_YOUR_BOOKS_CSV_PATH"
ratings_path = "HERE_GOES_YOUR_RATINGS_CSV_PATH"
users_path = "HERE_GOES_YOUR_USERS_CSV_PATH"

# Read CSV files using low_memory=False to avoid data type warning issues
books = pd.read_csv(books_path, low_memory=False)
ratings = pd.read_csv(ratings_path, low_memory=False)
users = pd.read_csv(users_path, low_memory=False)

print(f"Data Loaded Successfully!\nBooks: {books.shape}, Ratings: {ratings.shape}, Users: {users.shape}")


# ==========================================
# 2. DATA PREPROCESSING & CLEANING
# ==========================================

# Standardize missing values and column types
books.dropna(subset=['Book-Title', 'Book-Author'], inplace=True)
ratings.dropna(inplace=True)

# Merge ratings with book information to get the titles
ratings_with_name = ratings.merge(books, on='ISBN')


# ==========================================
# 3. FILTERING ACTIVE USERS AND POPULAR BOOKS
# ==========================================

# Filter users who have rated more than 200 books to keep reliable reviewers
user_counts = ratings_with_name.groupby('User-ID').count()['Book-Rating']
active_users = user_counts[user_counts > 200].index
filtered_rating = ratings_with_name[ratings_with_name['User-ID'].isin(active_users)]

# Filter books that have received at least 50 ratings to avoid obscure items
book_counts = filtered_rating.groupby('Book-Title').count()['Book-Rating']
famous_books = book_counts[book_counts >= 50].index
final_ratings = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]


# ==========================================
# 4. CREATE USER-ITEM MATRIX & COSIM SIMILARITY
# ==========================================

# Pivot table creates a matrix of [Book-Titles x User-IDs] filled with ratings
pivot_table = final_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')
pivot_table.fillna(0, inplace=True)

# Calculate Cosine Similarity score among books
similarity_scores = cosine_similarity(pivot_table)


# ==========================================
# 5. DEFINE THE RECOMMENDATION FUNCTION
# ==========================================

def recommend_books(book_name):
    """
    Takes a book title as input, finds similar books based on 
    Cosine Similarity, and prints the top 5 recommendations.
    """
    # Check if the book exists in our model matrix
    if book_name not in pivot_table.index:
        print(f"'{book_name}' is not found in the matrix index. Try another popular book!")
        return
    
    # Fetch index position of the input book
    index = np.where(pivot_table.index == book_name)[0][0]
    
    # Sort the similarity scores in descending order and fetch top 5 items
    similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:6]
    
    print(f"\nTop 5 Recommendations for '{book_name}':")
    print("-" * 50)
    for i in similar_items:
        print(pivot_table.index[i[0]])

# ==========================================
# 6. TEST THE MODEL
# ==========================================

# Example Test Case (Make sure to use an exact book title from the dataset)
try:
    sample_book = pivot_table.index[0]  # Gets the very first book title alphabetically
    recommend_books(sample_book)
except Exception as e:
    print("Error during recommendation test:", e)

# ==========================================
# 7. INSTALL & BUILD GRADIO UI INTERFACE
# ==========================================

# Install gradio directly inside the notebook environment
!pip install -q gradio

import gradio as gr

def gradio_recommend(book_name):
    """
    Wrapper function for Gradio that takes a book name 
    and returns the top 5 recommended books as text.
    """
    if book_name not in pivot_table.index:
        return "Book not found in our database. Please select another book from the list."
    
    # Find the index of the book
    index = np.where(pivot_table.index == book_name)[0][0]
    
    # Get top 5 similar items
    similar_items = sorted(list(enumerate(similarity_scores[index])), key=lambda x: x[1], reverse=True)[1:6]
    
    # Format the recommendations into a neat numbered list
    recommendations = []
    for i in similar_items:
        recommendations.append(pivot_table.index[i[0]])
        
    return "\n".join([f"{idx+1}. {book}" for idx, book in enumerate(recommendations)])

# Get a sorted list of all available books for the dropdown menu
available_books = sorted(list(pivot_table.index))

# Create the Gradio interface layout
demo = gr.Interface(
    fn=gradio_recommend,
    inputs=gr.Dropdown(
        choices=available_books, 
        label="Select a Book you Like", 
        value="1984" # Default selection
    ),
    outputs=gr.Textbox(label="Top 5 Recommended Books for You:", lines=6),
    title="📚 Smart Book Recommendation System",
    description="Select a book from the dropdown list, and the AI model will instantly recommend 5 similar books based on community reading patterns.",
    theme="soft"
)

# Launch the web interface inside the notebook with a shareable public link
demo.launch(share=True)
